# SOC L2 Lab — Análisis Forense y Correlación de Logs del Laboratorio

Este notebook está diseñado para el **Analista SOC L2** para investigar, analizar y visualizar en profundidad la actividad y telemetría generada en el entorno por el script `Invoke-SocLabLogGeneration.ps1`.

### Cobertura de Fuentes Analizadas:
* **Windows Endpoint & Active Directory (P02, P03, P14, P15):** Eventos `4625` (Fuerza bruta), `4688` (Procesos sospechosos), `4720`/`4728` (Manipulación de usuarios/grupos AD).
* **Linux Server & Aplicaciones (P04, P04B, P11, P13):** `Syslog` (sshd, sudo, su), logs de Apache y autenticación PostgreSQL.
* **Servicios Web & PaaS (P05, P12):** `AppServiceHTTPLogs` (escaneo web) y `W3CIISLog` (IIS en Windows).
* **Seguridad en la Nube & Kubernetes (P07, P08, P09, P10):** `AZKVAuditLogs` (Key Vault), `AKSAuditAdmin` (K8s API), `ContainerLogV2` y `AzureActivity`.

---

In [ ]:
# Instalación de librerías requeridas en Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


## 1. Inicialización y Conexión con Microsoft Sentinel / Log Analytics

Utiliza la Identidad Administrada de la Compute Instance (`soc-l2-ci`) para autenticarse automáticamente mediante `DefaultAzureCredential` contra el workspace `soc-l2-law`.

In [ ]:
# Configuración del entorno del laboratorio
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'  # customerId de soc-l2-law
LOOKBACK_HOURS = 48

from datetime import timedelta
import pandas as pd
import matplotlib.pyplot as plt
import json
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

# Configurar visualización de DataFrames
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

def query_sentinel(kql_query, hours=LOOKBACK_HOURS):
    """Ejecuta una consulta KQL en Log Analytics y retorna un DataFrame de Pandas."""
    resp = client.query_workspace(WORKSPACE_ID, kql_query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        table = resp.partial_data[0]
    elif resp.status == LogsQueryStatus.SUCCESS:
        if not resp.tables:
            return pd.DataFrame()
        table = resp.tables[0]
    else:
        raise RuntimeError(f"Error consultando Log Analytics: {resp}")
    return pd.DataFrame(data=table.rows, columns=table.columns)

print(f"[✓] Conector LogsQueryClient inicializado para el Workspace: {WORKSPACE_ID}")

## 2. Resumen General de Telemetría Ingerida

Consolidación de todos los registros que contienen marcadores `SOC-L2` para verificar la visibilidad en cada tabla.

In [ ]:
kql_resumen = '''
let win = SecurityEvent 
    | where (CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2-p02' or TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15')
    | summarize Total_Eventos = count() by Tabla = 'SecurityEvent (Windows/AD)';
let sys = Syslog 
    | where SyslogMessage has 'SOC-L2' or SyslogMessage has 'socp04' or SyslogMessage has 'soc_l2_p13'
    | summarize Total_Eventos = count() by Tabla = 'Syslog (Linux/Servicios)';
let app = AppServiceHTTPLogs 
    | where UserAgent startswith 'SOC-L2-Lab' 
    | summarize Total_Eventos = count() by Tabla = 'AppServiceHTTPLogs';
let iis = W3CIISLog 
    | where csUserAgent startswith 'SOC-L2-Lab' or csUriStem has 'soc-l2'
    | summarize Total_Eventos = count() by Tabla = 'W3CIISLog (IIS)';
let kv = AZKVAuditLogs 
    | where RequestUri has 'soc-l2' or OperationName has 'Secret'
    | summarize Total_Eventos = count() by Tabla = 'AZKVAuditLogs (Key Vault)';
let aks_audit = AKSAuditAdmin 
    | where RequestUri has 'soc-l2'
    | summarize Total_Eventos = count() by Tabla = 'AKSAuditAdmin (K8s API)';
let aks_pods = ContainerLogV2 
    | where LogMessage has 'SOC-L2'
    | summarize Total_Eventos = count() by Tabla = 'ContainerLogV2 (Pods AKS)';
let act = AzureActivity 
    | where Properties has 'SOCPrueba' or OperationNameValue has 'Microsoft.Resources/tags'
    | summarize Total_Eventos = count() by Tabla = 'AzureActivity (ARM)';
union win, sys, app, iis, kv, aks_audit, aks_pods, act
| sort by Total_Eventos desc
'''

df_resumen = query_sentinel(kql_resumen)
display(df_resumen)

# Gráfico de distribución de eventos por tabla
if not df_resumen.empty:
    plt.figure(figsize=(10, 4))
    plt.barh(df_resumen['Tabla'], df_resumen['Total_Eventos'], color='#0078D4')
    plt.xlabel('Cantidad de Eventos')
    plt.title('Distribución de Eventos del Laboratorio por Tabla')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 3. Investigación Forense de Windows (P02, P03, P15)

### 3.1. Prueba P02: Fuerza Bruta / Fallos de Autenticación (EventID 4625)
Analiza intentos de acceso con cuentas inexistentes marcadas y clasifica por tipo de inicio de sesión.

In [ ]:
kql_p02 = '''
SecurityEvent
| where EventID == 4625 and TargetUserName startswith 'soc-l2-p02'
| extend LogonTypeDesc = case(
    LogonType == 2, 'Interactive (Consola)',
    LogonType == 3, 'Network (SMB/Red)',
    LogonType == 10, 'RemoteInteractive (RDP)',
    tostring(LogonType)
  )
| summarize 
    Intentos_Fallidos = count(),
    Primer_Intento = min(TimeGenerated),
    Ultimo_Intento = max(TimeGenerated)
  by Computer, TargetUserName, LogonTypeDesc, IpAddress, SubStatus
| sort by Intentos_Fallidos desc
'''
df_p02 = query_sentinel(kql_p02)
print(f"Total de vectores de autenticación fallida P02: {len(df_p02)}")
display(df_p02)

### 3.2. Prueba P03: Ejecución de Procesos y LOLBAS (EventID 4688)
Inspecciona la creación de procesos con líneas de comando marcadas ejecutadas sobre la VM Windows.

In [ ]:
kql_p03 = r'''
SecurityEvent
| where EventID == 4688 and CommandLine has 'SOC-L2-P03'
| extend Proceso = tostring(split(NewProcessName, @'\')[-1])
| extend ProcesoPadre = tostring(split(ParentProcessName, @'\')[-1])
| project TimeGenerated, Computer, SubjectUserName, Proceso, ProcesoPadre, CommandLine
| sort by TimeGenerated desc
'''
df_p03 = query_sentinel(kql_p03)
display(df_p03)

### 3.3. Prueba P15: Persistencia y Gestión de Cuentas en AD DS (EventIDs 4720 a 4738)
Reconstruye la cadena cronológica de creación, activación, membresía en grupos y borrado de cuentas de prueba en Active Directory.

In [ ]:
kql_p15 = '''
SecurityEvent
| where EventID in (4720, 4722, 4724, 4725, 4726, 4727, 4728, 4729, 4730, 4732, 4733, 4738)
| where TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15'
| extend Accion = case(
    EventID == 4720, 'Usuario Creado',
    EventID == 4722, 'Usuario Habilitado',
    EventID == 4724, 'Password Reseteado',
    EventID == 4728, 'Agregado a Grupo Global',
    EventID == 4732, 'Agregado a Grupo Local',
    EventID == 4726, 'Usuario Eliminado',
    EventID == 4730, 'Grupo Eliminado',
    Activity
  )
| project TimeGenerated, EventID, Accion, SubjectUserName, TargetUserName, MemberName
| sort by TimeGenerated asc
'''
df_p15 = query_sentinel(kql_p15)
display(df_p15)

## 4. Investigación de Linux y Servidores Web (P04, P04B, P11, P12, P13)

### 4.1. Intentos de Acceso y Comandos en Linux (`Syslog`)
Eventos de elevación de privilegios `sudo`, accesos `sshd` fallidos e invocaciones con usuarios inexistentes.

In [ ]:
kql_linux = '''
Syslog
| where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd', 'su', 'sudo') and SyslogMessage has 'socp04')
| project TimeGenerated, Computer, Facility, SeverityLevel, ProcessName, SyslogMessage
| sort by TimeGenerated desc
'''
df_linux = query_sentinel(kql_linux)
display(df_linux)

### 4.2. Tráfico en Servidores Web: Apache (Linux) e IIS (Windows) - P11 y P12
Comparación de peticiones HTTP enviadas a Apache (`local6` en Syslog) y a IIS (`W3CIISLog`).

In [ ]:
kql_web = '''
let apache = Syslog
    | where Facility == 'local6' and SyslogMessage has 'SOC-L2-P11'
    | project TimeGenerated, Servidor = 'Apache (Linux)', Metodo = 'HTTP', Codigo = '200', Mensaje = SyslogMessage;
let iis = W3CIISLog
    | where csUserAgent startswith 'SOC-L2-Lab' or csUriStem has 'soc-l2-p12'
    | project TimeGenerated, Servidor = 'IIS (Windows)', Metodo = csMethod, Codigo = tostring(scStatus), Mensaje = strcat(csUriStem, ' | IP: ', cIP);
union apache, iis
| sort by TimeGenerated desc
'''
df_web = query_sentinel(kql_web)
display(df_web)

## 5. Auditoría de Seguridad Cloud: App Service, Key Vault y AKS (P05, P07, P08, P10)

### 5.1. P07: Accesos a Secretos en Azure Key Vault (`AZKVAuditLogs`)
Verifica lecturas autorizadas de secretos (`200 OK`) y sondeos de secretos inexistentes (`404 Not Found`).

In [ ]:
kql_kv = '''
AZKVAuditLogs
| where RequestUri has 'soc-l2-p07' or OperationName has 'Secret'
| project TimeGenerated, OperationName, ResultType, ResultSignature, CallerIpAddress, RequestUri
| sort by TimeGenerated desc
'''
df_kv = query_sentinel(kql_kv)
display(df_kv)


### 5.2. P08 y P10: Auditoría de Control Plane y Pods en AKS (`AKSAuditAdmin` y `ContainerLogV2`)

In [ ]:
kql_aks = '''
AKSAuditAdmin
| where RequestUri has 'soc-l2-p08'
| project TimeGenerated, Verb, ResponseStatus = parse_json(ResponseStatus).code, RequestUri, Usuario = tostring(parse_json(tostring(User)).username)
| sort by TimeGenerated desc
'''
df_aks = query_sentinel(kql_aks)
display(df_aks)

## 6. Matriz de Resumen y Exportación de Hallazgos para el Analista L2

Genera una tabla de incidentes/evidencias sintetizada mapeada contra el marco MITRE ATT&CK.

In [ ]:
resumen_hallazgos = [
    {'Prueba': 'P02', 'Vector': 'Windows Brute Force', 'Tactica_MITRE': 'Credential Access (T1110)', 'Registros': len(df_p02), 'Estado': 'Detectado' if len(df_p02) > 0 else 'Pendiente'},
    {'Prueba': 'P03', 'Vector': 'LOLBAS Execution', 'Tactica_MITRE': 'Execution (T1059.001)', 'Registros': len(df_p03), 'Estado': 'Detectado' if len(df_p03) > 0 else 'Pendiente'},
    {'Prueba': 'P15', 'Vector': 'AD Account Manipulation', 'Tactica_MITRE': 'Persistence / Privilege Escalation (T1098/T1078)', 'Registros': len(df_p15), 'Estado': 'Detectado' if len(df_p15) > 0 else 'Pendiente'},
    {'Prueba': 'P04/P04B', 'Vector': 'Linux Auth Failures', 'Tactica_MITRE': 'Initial Access / Lateral Movement (T1078)', 'Registros': len(df_linux), 'Estado': 'Detectado' if len(df_linux) > 0 else 'Pendiente'},
    {'Prueba': 'P11/P12', 'Vector': 'Web Probes (Apache/IIS)', 'Tactica_MITRE': 'Reconnaissance (T1595)', 'Registros': len(df_web), 'Estado': 'Detectado' if len(df_web) > 0 else 'Pendiente'},
    {'Prueba': 'P07', 'Vector': 'Key Vault Access & 404', 'Tactica_MITRE': 'Credential Access (T1555)', 'Registros': len(df_kv), 'Estado': 'Detectado' if len(df_kv) > 0 else 'Pendiente'},
    {'Prueba': 'P08/P10', 'Vector': 'AKS API & Pod Activity', 'Tactica_MITRE': 'Execution in Container (T1610)', 'Registros': len(df_aks), 'Estado': 'Detectado' if len(df_aks) > 0 else 'Pendiente'}
]

df_resumen_final = pd.DataFrame(resumen_hallazgos)
print("=== REPORTE SINTÉTICO DE CASOS DEL LABORATORIO (SOC L2) ===")
display(df_resumen_final)

# Guardar reporte localmente en CSV
df_resumen_final.to_csv('reporte_evidencias_soc_l2.csv', index=False)
print("[✓] Reporte de evidencias exportado con éxito a 'reporte_evidencias_soc_l2.csv'")